# ATP Historical Data Exploration
Using Jeff Sackmann's dataset — match results from 1968–2025.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('../data/tennis_atp')
sns.set_theme(style='whitegrid')

## 1. Load recent match data (2010–2025)

In [ ]:
years = range(2010, 2026)
frames = []
for yr in years:
    path = DATA_DIR / f'atp_matches_{yr}.csv'
    if path.exists():
        frames.append(pd.read_csv(path, low_memory=False))

df = pd.concat(frames, ignore_index=True)
df['tourney_date'] = pd.to_datetime(df['tourney_date'], format='%Y%m%d')
print(f'Matches loaded: {len(df):,}')
print(f'Date range   : {df.tourney_date.min().date()} → {df.tourney_date.max().date()}')
df.head(3)

In [ ]:
# What columns do we have?
print(list(df.columns))

## 2. Surface breakdown

In [ ]:
df['surface'].value_counts().plot(kind='bar', title='Matches by surface')
plt.tight_layout()

## 3. Ranking-based win rate
Core hypothesis: higher-ranked player wins more often.

In [ ]:
ranked = df.dropna(subset=['winner_rank', 'loser_rank']).copy()
ranked['winner_rank'] = ranked['winner_rank'].astype(int)
ranked['loser_rank']  = ranked['loser_rank'].astype(int)
ranked['higher_ranked_won'] = ranked['winner_rank'] < ranked['loser_rank']

print(f'Higher-ranked player win rate: {ranked["higher_ranked_won"].mean():.1%}')

# Break down by surface
ranked.groupby('surface')['higher_ranked_won'].mean().sort_values(ascending=False).plot(
    kind='bar', title='Higher-ranked win rate by surface', ylim=(0.5, 0.8)
)
plt.tight_layout()

## 4. Rank differential vs win probability

In [ ]:
ranked['rank_diff'] = ranked['loser_rank'] - ranked['winner_rank']  # positive = winner was higher ranked

# Bin rank differences and compute win rates
bins = [-500, -100, -50, -20, -10, -5, 0, 5, 10, 20, 50, 100, 500]
labels = [str(b) for b in bins[:-1]]
ranked['diff_bin'] = pd.cut(ranked['rank_diff'], bins=bins, labels=labels)

# P(higher-ranked wins) by bin — when rank_diff > 0 winner was better ranked
ranked['p1_won'] = (ranked['rank_diff'] > 0).astype(int)
win_by_diff = ranked.groupby('diff_bin', observed=True)['p1_won'].mean()
win_by_diff.plot(kind='bar', title='P(better-ranked player wins) by rank gap', ylim=(0, 1))
plt.axhline(0.5, color='red', linestyle='--')
plt.tight_layout()

## 5. Head-to-head records
How often does H2H record predict the winner?

In [ ]:
# Build cumulative H2H table
df_sorted = df.sort_values('tourney_date')

def get_h2h(p1, p2, before_date, data):
    hist = data[
        (data['tourney_date'] < before_date) &
        (
            ((data['winner_name'] == p1) & (data['loser_name'] == p2)) |
            ((data['winner_name'] == p2) & (data['loser_name'] == p1))
        )
    ]
    p1_wins = (hist['winner_name'] == p1).sum()
    p2_wins = (hist['winner_name'] == p2).sum()
    return p1_wins, p2_wins

# Sample 500 matches from 2022+ with ranked players
sample = ranked[ranked['tourney_date'].dt.year >= 2022].sample(500, random_state=42)

results = []
for _, row in sample.iterrows():
    w, l = row['winner_name'], row['loser_name']
    w_h2h, l_h2h = get_h2h(w, l, row['tourney_date'], df_sorted)
    results.append({
        'winner_h2h': w_h2h,
        'loser_h2h':  l_h2h,
        'h2h_favored_winner': w_h2h > l_h2h,
        'h2h_available': (w_h2h + l_h2h) > 0,
    })

h2h_df = pd.DataFrame(results)
has_history = h2h_df[h2h_df['h2h_available']]
print(f'Matches with H2H history: {len(has_history)} / {len(sample)}')
print(f'H2H leader wins: {has_history["h2h_favored_winner"].mean():.1%}')

## 6. Recent form (last N matches win rate)

In [ ]:
def recent_win_rate(player, before_date, data, n=10):
    recent = data[
        (data['tourney_date'] < before_date) &
        ((data['winner_name'] == player) | (data['loser_name'] == player))
    ].tail(n)
    if len(recent) == 0:
        return None
    return (recent['winner_name'] == player).mean()

# Sample and compute form diff
sample2 = ranked[ranked['tourney_date'].dt.year >= 2022].sample(300, random_state=1)
form_results = []
for _, row in sample2.iterrows():
    wf = recent_win_rate(row['winner_name'], row['tourney_date'], df_sorted)
    lf = recent_win_rate(row['loser_name'],  row['tourney_date'], df_sorted)
    if wf is not None and lf is not None:
        form_results.append({'better_form_won': wf > lf})

form_df = pd.DataFrame(form_results)
print(f'Player with better recent form wins: {form_df["better_form_won"].mean():.1%}')

## 7. Surface-specific win rate per player

In [ ]:
# Top 20 active players surface win rates
recent = df[df['tourney_date'].dt.year >= 2020]

all_players = pd.concat([
    recent[['winner_name']].rename(columns={'winner_name':'player'}),
    recent[['loser_name']].rename(columns={'loser_name':'player'})
])['player'].value_counts().head(30).index.tolist()

rows = []
for p in all_players:
    for surf in ['Hard', 'Clay', 'Grass']:
        surf_df = recent[recent['surface'] == surf]
        wins   = (surf_df['winner_name'] == p).sum()
        losses = (surf_df['loser_name']  == p).sum()
        if wins + losses >= 5:
            rows.append({'player': p, 'surface': surf, 'win_rate': wins / (wins + losses), 'matches': wins + losses})

surf_df2 = pd.DataFrame(rows).pivot(index='player', columns='surface', values='win_rate')
sns.heatmap(surf_df2.round(2), annot=True, fmt='.0%', cmap='RdYlGn', vmin=0.3, vmax=0.8)
plt.title('Surface win rates (2020–2025)')
plt.tight_layout()

## 8. Load Elo ratings

In [ ]:
# Sackmann also publishes Elo ratings
elo_path = DATA_DIR / 'atp_players.csv'
players = pd.read_csv(elo_path)
print(f'Players in database: {len(players)}')
players.head()